In [16]:
# Imports and Paths

import pandas as pd
from pathlib import Path

project_root = Path.cwd().parent
data_dir = project_root / "data"
raw_dir = data_dir / "raw"
processed_dir = data_dir / "processed"
figures_dir = project_root / "figures"

processed_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

In [17]:
# Drug File: Inspect

drug_path = raw_dir / "drug.csv"

# Load just enough to see column names
df_drug_sample = pd.read_csv(drug_path, nrows=5)
df_drug_sample.columns.tolist()

['DATASET',
 'NLME_RESULT_ID',
 'NLME_CURVE_ID',
 'CELL_LINE_NAME',
 'SANGER_MODEL_ID',
 'CANCER_TYPE',
 'DRUG_ID',
 'DRUG_NAME',
 'PUTATIVE_TARGET',
 'PATHWAY_NAME',
 'MIN_CONC',
 'MAX_CONC',
 'LN_IC50',
 'AUC',
 'RMSE',
 'Z_SCORE']

In [18]:
# Drug File: Rename
df_drug = pd.read_csv(drug_path)

df_drug = df_drug.rename(columns={
    "DRUG_NAME": "drug_name",
    "SANGER_MODEL_ID": "model_id",
    "LN_IC50": "ln_ic50"
})

df_drug[["drug_name", "model_id", "ln_ic50"]].head()

,drug_name,model_id,ln_ic50
0,Camptothecin,SIDM01132,-1.463887
1,Camptothecin,SIDM00848,-4.869455
2,Camptothecin,SIDM00263,-3.360586
3,Camptothecin,SIDM00269,-5.044940
4,Camptothecin,SIDM00203,-3.741991


In [19]:
# Expression File: Inspect a Sample and Standardize Names

expr_path = raw_dir / "expr.csv"

df_expr_sample = pd.read_csv(expr_path, nrows=5)
df_expr_sample.columns.tolist()

['dataset_id',
 'model_id',
 'gene_id',
 'gene_symbol',
 'ensembl_gene_id',
 'model_name',
 'htseq_read_count',
 'rsem_expected_count',
 'rsem_fpkm',
 'rsem_tpm',
 'htseq_fpkm',
 'data_source',
 'duplicate']

In [20]:
# Get Models of Interest (Cisplatin)

drug_col = "drug_name"
model_col = "model_id"
response_col = "ln_ic50"

df_cis = df_drug[df_drug[drug_col] == "Cisplatin"].copy()
df_cis = df_cis[[model_col, drug_col, response_col]].dropna()

models_of_interest = set(df_cis[model_col].unique())
assert df_cis[model_col].is_unique
len(models_of_interest), df_cis.head()

(760,
        model_id  drug_name   ln_ic50
 1709  SIDM01132  Cisplatin  3.168266
 1710  SIDM00848  Cisplatin -0.007561
 1711  SIDM00269  Cisplatin  0.045420
 1712  SIDM00203  Cisplatin  0.923136
 1713  SIDM01111  Cisplatin -0.134330)

In [6]:
# Chunk‑Load Expression

value_col_original = "rsem_tpm"

chunks = []
total_rows = 0
kept_rows = 0

for i, chunk in enumerate(pd.read_csv(
    expr_path,
    chunksize=50_000,
    usecols=["model_id", "gene_symbol", value_col_original])):

  # Rename columns inside chunk
    chunk = chunk.rename(columns={
        value_col_original: "expr_value",
    })

    total_rows += len(chunk)

    # Filter to models of interest
    chunk = chunk[chunk["model_id"].isin(models_of_interest)]
    kept_rows += len(chunk)

    if not chunk.empty:
        chunks.append(chunk)

    # Progress message every few chunks
    if i % 10 == 0:
        print(f"Chunk {i}: total_rows={total_rows}, kept_rows={kept_rows}")

df_expr_small = pd.concat(chunks, ignore_index=True)

df_expr_small.head()

Chunk 0: total_rows=50000, kept_rows=18742
Chunk 10: total_rows=550000, kept_rows=206164
Chunk 20: total_rows=1050000, kept_rows=393616
Chunk 30: total_rows=1550000, kept_rows=581088
Chunk 40: total_rows=2050000, kept_rows=768546
Chunk 50: total_rows=2550000, kept_rows=955974
Chunk 60: total_rows=3050000, kept_rows=1143411
Chunk 70: total_rows=3550000, kept_rows=1330876
Chunk 80: total_rows=4050000, kept_rows=1518352
Chunk 90: total_rows=4550000, kept_rows=1705752
Chunk 100: total_rows=5050000, kept_rows=1893225
Chunk 110: total_rows=5550000, kept_rows=2080681
Chunk 120: total_rows=6050000, kept_rows=2268121
Chunk 130: total_rows=6550000, kept_rows=2455568
Chunk 140: total_rows=7050000, kept_rows=2643028
Chunk 150: total_rows=7550000, kept_rows=2830454
Chunk 160: total_rows=8050000, kept_rows=3017925
Chunk 170: total_rows=8550000, kept_rows=3205347
Chunk 180: total_rows=9050000, kept_rows=3392824
Chunk 190: total_rows=9550000, kept_rows=3580266
Chunk 200: total_rows=10050000, kept_rows

,model_id,gene_symbol,expr_value
0,SIDM00776,DVL2,3.5521
1,SIDM00437,STPG1,2.2898
2,SIDM00598,STPG1,2.9486
3,SIDM00657,STPG1,1.3674
4,SIDM00506,STPG1,4.4874


In [7]:
# Save filtered table

reduced_expr_path = processed_dir / "rnaseq_expression_cisplatin_models.csv"
df_expr_small.to_csv(reduced_expr_path, index=False)
reduced_expr_path

WindowsPath('C:/Users/draar/gdsc-drug-response-project/data/processed/rnaseq_expression_cisplatin_models.csv')

In [12]:
# Reload filtered table (you can restart the notebook from here without redoing the chunk-loading)

df_expr_small = pd.read_csv(processed_dir / "rnaseq_expression_cisplatin_models.csv")

In [13]:
# Pivot expression to wide format and merge

df_expr_wide = (
    df_expr_small.pivot_table(
        index=model_col,
        columns="gene_symbol",
        values="expr_value",
        aggfunc="mean"
    ).reset_index()
)

df_expr_wide.shape, df_expr_wide.head()

((743, 41144),
 gene_symbol   model_id    A1BG  A1BG-AS1    A1CF     A2M  A2M-AS1   A2ML1  \
 0            SIDM00003  3.9736    3.1408  0.0144  4.1326   0.4114  0.0000   
 1            SIDM00042  0.5059    0.8639  0.0566  0.2265   1.3561  0.4542   
 2            SIDM00044  5.3125    3.5398  0.0144  0.0000   0.0144  0.0000   
 3            SIDM00045  0.0976    0.1890  2.9982  0.0000   0.0286  0.0000   
 4            SIDM00047  2.9782    1.5211  0.0144  0.0000   1.1375  0.0841   
 
 gene_symbol  A2ML1-AS1  A2ML1-AS2   A2MP1  ...    ZXDA    ZXDB    ZXDC  \
 0                  0.0        0.0  0.0000  ...  1.0909  2.1010  4.0488   
 1                  0.0        0.0  0.0286  ...  1.2630  2.5008  4.7840   
 2                  0.0        0.0  0.0566  ...  1.1177  2.2780  4.2787   
 3                  0.0        0.0  0.0000  ...  1.5945  3.7761  5.0781   
 4                  0.0        0.0  0.0000  ...  1.2690  2.3646  3.9079   
 
 gene_symbol  ZYG11A  ZYG11AP1  ZYG11B     ZYX  ZYXP1   ZZEF1  

In [14]:
# Merge with Cisplatin response

df_merged = pd.merge(
    df_cis,
    df_expr_wide,
    how="inner",
    on=model_col
)

df_merged.shape, df_merged.head()

((743, 41146),
     model_id  drug_name   ln_ic50     A1BG  A1BG-AS1    A1CF      A2M  \
 0  SIDM01132  Cisplatin  3.168266  5.18545    2.9833  0.0000  0.52325   
 1  SIDM00848  Cisplatin -0.007561  4.86890    3.4476  0.0000  1.99640   
 2  SIDM00269  Cisplatin  0.045420  4.54970    3.8001  0.0841  3.21570   
 3  SIDM00203  Cisplatin  0.923136  4.22500    3.2095  0.0144  0.50590   
 4  SIDM01111  Cisplatin -0.134330  4.49250    2.8115  0.0144  0.47510   
 
    A2M-AS1   A2ML1  A2ML1-AS1  ...     ZXDA     ZXDB     ZXDC  ZYG11A  \
 0   0.0000  0.0213        0.0  ...  0.31285  1.60475  3.10825  0.6801   
 1   0.1243  3.1890        0.0  ...  1.68260  2.33060  5.18630  0.0841   
 2   0.0566  0.5753        0.0  ...  2.82980  3.41280  6.78270  2.3190   
 3   0.0976  2.8156        0.0  ...  2.49570  3.47640  5.30710  3.0321   
 4   0.0286  0.5558        0.0  ...  1.28690  2.26000  4.94810  2.7634   
 
    ZYG11AP1   ZYG11B      ZYX  ZYXP1    ZZEF1    ZZZ3  
 0       0.0  2.58955  5.95435    0.

In [15]:
# Save for later notebooks

out_path = processed_dir / "cisplatin_rnaseq_modelready.csv"
print("Saving to:", out_path)
df_merged.to_csv(out_path, index=False)

Saving to: C:\Users\draar\gdsc-drug-response-project\data\processed\cisplatin_rnaseq_modelready.csv
